# LeetCode 3327 — Check if DFS Strings Are Palindromes bằng Python

- Link: https://leetcode.com/problems/check-if-dfs-strings-are-palindromes/
- Độ khó: **Hard**
- Chủ đề: **Duyệt hậu thứ tự (post-order)** — chuỗi của mỗi cây con là một **đoạn liên tiếp** trong chuỗi của gốc; kiểm tra palindrome đoạn con bằng **Manacher** hoặc **băm đa thức**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findAnswer(self, parent: List[int], s: str) -> List[bool]:
```

## 1. Mục tiêu học

1. Nhận ra `dfs(x)` là duyệt **hậu thứ tự** cây con của `x` (con theo thứ tự tăng, rồi mới tới `x`).
2. Vì `dfsStr` dùng chung, chuỗi `dfs(x)` chính là **đoạn liên tiếp** `[tin[x], tout[x]]` trong chuỗi `T = dfs(0)`, với `tout[x]` = vị trí của `x` (ký tự cuối) và `tin[x] = tout[x] − size[x] + 1`.
3. Quy về `n` truy vấn "đoạn con của `T` có là palindrome?" ⇒ **Manacher** `O(n)` hoặc **băm** `O(n)`.
4. Duyệt cây **không đệ quy** (cây có thể là đường thẳng sâu `10^5`).
5. Kiểm chứng bằng vét cạn gọi `dfs(i)` cho từng `i`.

## 2. Đề bài & ví dụ

Cây gốc `0` cho bởi mảng `parent`, mỗi nút `i` mang ký tự `s[i]`. Hàm `dfs(x)`: duyệt các con `y` của `x` theo **thứ tự số hiệu tăng dần**, gọi `dfs(y)`; sau đó **thêm `s[x]`** vào cuối chuỗi chung `dfsStr`. Với mỗi `i`, làm rỗng `dfsStr`, gọi `dfs(i)` và cho biết chuỗi thu được có là palindrome không.

**Ví dụ 1:** `parent = [-1,0,0,1,1,2]`, `s = "aababa"` → `[true,true,false,true,true,true]` (`dfs(0) = "abaaba"`, `dfs(1) = "aba"`, `dfs(2) = "ab"`).
**Ví dụ 2:** `parent = [-1,0,0,0,0]`, `s = "aabcb"` → toàn `true`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `parent[0] = −1`, `0 ≤ parent[i] ≤ n − 1`; `s` gồm chữ thường.

In [1]:
from typing import List

examples = [([-1, 0, 0, 1, 1, 2], "aababa", [True, True, False, True, True, True]),
            ([-1, 0, 0, 0, 0], "aabcb", [True] * 5)]
for parent, s, exp in examples:
    print(parent, repr(s), '-> kỳ vọng', exp)

[-1, 0, 0, 1, 1, 2] 'aababa' -> kỳ vọng [True, True, False, True, True, True]
[-1, 0, 0, 0, 0] 'aabcb' -> kỳ vọng [True, True, True, True, True]


## 3. Kiến thức nền

### 3.1. Danh sách con theo thứ tự tăng

Duyệt `i = 1..n−1` và thêm `i` vào `children[parent[i]]` ⇒ mỗi danh sách con **tự động tăng dần**.

### 3.2. Hậu thứ tự không đệ quy

Dùng ngăn xếp các cặp `(nút, chỉ số con kế tiếp)`; khi đã duyệt hết con thì "thăm" nút (ghi ký tự).

In [2]:
def build_children(parent):
    n = len(parent)
    ch = [[] for _ in range(n)]
    for i in range(1, n):
        ch[parent[i]].append(i)
    return ch

def postorder(parent, s):
    """Trả về (T, tout, size): T = dfs(0), tout[x] = vị trí của x trong T, size[x] = kích thước cây con."""
    n = len(parent)
    ch = build_children(parent)
    T, tout, size = [], [0] * n, [1] * n
    stack = [(0, 0)]
    while stack:
        x, i = stack.pop()
        if i < len(ch[x]):
            stack.append((x, i + 1))
            stack.append((ch[x][i], 0))
        else:
            tout[x] = len(T)
            T.append(s[x])
            if x:
                size[parent[x]] += size[x]
    return ''.join(T), tout, size

T, tout, size = postorder(*examples[0][:2])
print('T =', T, '| tout =', tout, '| size =', size)
print('đoạn của nút 1:', T[tout[1] - size[1] + 1: tout[1] + 1], '| nút 2:', T[tout[2] - size[2] + 1: tout[2] + 1])

T = abaaba | tout = [5, 2, 4, 0, 1, 3] | size = [6, 3, 2, 1, 1, 1]
đoạn của nút 1: aba | nút 2: ab


Cộng `size[parent[x]] += size[x]` ngay khi thăm `x` là đúng vì trong hậu thứ tự, mọi nút con được thăm **trước** cha.

### 3.3. Manacher

Chèn ký tự phân cách: `t = "#" + "#".join(T) + "#"` (độ dài `2n + 1`). `P[c]` = bán kính palindrome lớn nhất tâm `c` trong `t`. Đoạn `T[l..r]` là palindrome ⇔ `P[l + r + 1] ≥ r − l + 1` (tâm của đoạn trong `t` là `l + r + 1`).

In [3]:
def manacher(T):
    t = '#' + '#'.join(T) + '#'
    m = len(t)
    P = [0] * m
    c = r = 0
    for i in range(m):
        if i < r:
            P[i] = min(r - i, P[2 * c - i])
        while i - P[i] - 1 >= 0 and i + P[i] + 1 < m and t[i - P[i] - 1] == t[i + P[i] + 1]:
            P[i] += 1
        if i + P[i] > r:
            c, r = i, i + P[i]
    return P

def is_pal(P, l, r):
    return P[l + r + 1] >= r - l + 1

P = manacher("abaaba")
print([(l, r, "abaaba"[l:r + 1], is_pal(P, l, r)) for l, r in [(0, 5), (1, 3), (0, 1), (2, 3)]])

[(0, 5, 'abaaba', True), (1, 3, 'baa', False), (0, 1, 'ab', False), (2, 3, 'aa', True)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: cây `0 → {1, 2}`, `1 → {3, 4}`, `2 → {5}`. Hậu thứ tự: `3, 4, 1, 5, 2, 0` ⇒ `T = s[3] s[4] s[1] s[5] s[2] s[0] = "abaaba"`.

- Cây con của `1` = `{3, 4, 1}` ⇒ đoạn `T[0..2] = "aba"` ✓ palindrome.
- Cây con của `2` = `{5, 2}` ⇒ đoạn `T[3..4] = "ab"` ✗.

Mọi cây con là một **khối liên tiếp** của hậu thứ tự, kết thúc tại chính nút đó.

## 5. Cách 1 — Brute Force: gọi `dfs(i)` cho từng `i`

In [4]:
def brute(parent, s):
    ch = build_children(parent)
    ans = []
    for i in range(len(parent)):
        out = []
        stack = [(i, 0)]
        while stack:
            x, j = stack.pop()
            if j < len(ch[x]):
                stack.append((x, j + 1)); stack.append((ch[x][j], 0))
            else:
                out.append(s[x])
        w = ''.join(out)
        ans.append(w == w[::-1])
    return ans

for parent, s, exp in examples:
    print(brute(parent, s), '| kỳ vọng', exp)

[True, True, False, True, True, True] | kỳ vọng [True, True, False, True, True, True]
[True, True, True, True, True] | kỳ vọng [True, True, True, True, True]


**Độ phức tạp:** `O(Σ size) = O(n · độ sâu)` — tới `O(n²)` với cây đường thẳng.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `T = dfs(0)`, `tout[x]`, `size[x]` bằng một lần hậu thứ tự lặp.
2. Chạy Manacher trên `T`.
3. `answer[x] = is_pal(tout[x] − size[x] + 1, tout[x])`.

Tổng `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Băm đa thức xuôi/ngược

`H[i]` = hash tiền tố của `T`, `R[i]` = hash tiền tố của `T` đảo ngược. Đoạn `[l, r]` palindrome ⇔ hash xuôi của `T[l..r]` = hash của đoạn tương ứng trong chuỗi đảo `[n−1−r, n−1−l]`. Dùng **hai modulo** để giảm xác suất va chạm.

In [5]:
def hash_solution(parent, s):
    T, tout, size = postorder(parent, s)
    n = len(T)
    mods, base = (10 ** 9 + 7, 998244353), 131
    res = [True] * n
    tables = []
    for M in mods:
        pw = [1] * (n + 1); H = [0] * (n + 1); R = [0] * (n + 1)
        rev = T[::-1]
        for i in range(n):
            pw[i + 1] = pw[i] * base % M
            H[i + 1] = (H[i] * base + ord(T[i])) % M
            R[i + 1] = (R[i] * base + ord(rev[i])) % M
        tables.append((M, pw, H, R))
    for x in range(n):
        r = tout[x]; l = r - size[x] + 1
        for M, pw, H, R in tables:
            fwd = (H[r + 1] - H[l] * pw[r - l + 1]) % M
            bl, br = n - 1 - r, n - 1 - l
            bwd = (R[br + 1] - R[bl] * pw[br - bl + 1]) % M
            if fwd != bwd:
                res[x] = False
                break
    return res

for parent, s, exp in examples:
    print(hash_solution(parent, s), '| kỳ vọng', exp)

[True, True, False, True, True, True] | kỳ vọng [True, True, False, True, True, True]
[True, True, True, True, True] | kỳ vọng [True, True, True, True, True]


### 7.2. Cách 3 — Manacher (lời giải chính)

In [6]:
def manacher_solution(parent, s):
    T, tout, size = postorder(parent, s)
    P = manacher(T)
    return [is_pal(P, tout[x] - size[x] + 1, tout[x]) for x in range(len(parent))]

for parent, s, exp in examples:
    print(manacher_solution(parent, s), '| kỳ vọng', exp)

[True, True, False, True, True, True] | kỳ vọng [True, True, False, True, True, True]
[True, True, True, True, True] | kỳ vọng [True, True, True, True, True]


### 7.3. Cách 4 — So sánh trực tiếp từng đoạn (`O(Σ size)`), nhưng dùng `T` chung

Đã có `T`, kiểm tra `T[l:r+1] == T[l:r+1][::-1]` cho từng nút. Vẫn `O(n · độ sâu)` nhưng hằng số nhỏ (phép so chuỗi chạy bằng C) — minh họa rằng **bước quy về đoạn con** mới là then chốt, còn Manacher/băm chỉ để bỏ phụ thuộc độ sâu.

In [7]:
def slice_solution(parent, s):
    T, tout, size = postorder(parent, s)
    out = []
    for x in range(len(parent)):
        w = T[tout[x] - size[x] + 1: tout[x] + 1]
        out.append(w == w[::-1])
    return out

for parent, s, exp in examples:
    print(slice_solution(parent, s), '| kỳ vọng', exp)

[True, True, False, True, True, True] | kỳ vọng [True, True, False, True, True, True]
[True, True, True, True, True] | kỳ vọng [True, True, True, True, True]


## 8. TRACE từng bước

Trace trên ví dụ 1: in hậu thứ tự, đoạn của từng nút và bán kính Manacher tại tâm của đoạn.

In [8]:
def trace(parent, s):
    T, tout, size = postorder(parent, s)
    P = manacher(T)
    order = sorted(range(len(parent)), key=lambda x: tout[x])
    print('hậu thứ tự:', order, '-> T =', T)
    for x in range(len(parent)):
        r = tout[x]; l = r - size[x] + 1
        print(f"  nút {x}: đoạn [{l}, {r}] = {T[l:r + 1]!r:8s} tâm {l + r + 1:2d}, P = {P[l + r + 1]}, "
              f"cần >= {r - l + 1} -> {is_pal(P, l, r)}")
    print('vét cạn:', brute(parent, s))

trace(*examples[0][:2])

hậu thứ tự: [3, 4, 1, 5, 2, 0] -> T = abaaba
  nút 0: đoạn [0, 5] = 'abaaba' tâm  6, P = 6, cần >= 6 -> True
  nút 1: đoạn [0, 2] = 'aba'    tâm  3, P = 3, cần >= 3 -> True
  nút 2: đoạn [3, 4] = 'ab'     tâm  8, P = 0, cần >= 2 -> False
  nút 3: đoạn [0, 0] = 'a'      tâm  1, P = 1, cần >= 1 -> True
  nút 4: đoạn [1, 1] = 'b'      tâm  3, P = 3, cần >= 1 -> True
  nút 5: đoạn [3, 3] = 'a'      tâm  7, P = 1, cần >= 1 -> True
vét cạn: [True, True, False, True, True, True]


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def findAnswer(self, parent: List[int], s: str) -> List[bool]:
        n = len(parent)
        ch = [[] for _ in range(n)]
        for i in range(1, n):
            ch[parent[i]].append(i)
        T = []
        tout = [0] * n
        size = [1] * n
        stack = [(0, 0)]
        while stack:
            x, i = stack.pop()
            if i < len(ch[x]):
                stack.append((x, i + 1))
                stack.append((ch[x][i], 0))
            else:
                tout[x] = len(T)
                T.append(s[x])
                if x:
                    size[parent[x]] += size[x]
        t = '#' + '#'.join(T) + '#'
        m = len(t)
        P = [0] * m
        c = r = 0
        for i in range(m):
            p = min(r - i, P[2 * c - i]) if i < r else 0
            while i - p - 1 >= 0 and i + p + 1 < m and t[i - p - 1] == t[i + p + 1]:
                p += 1
            P[i] = p
            if i + p > r:
                c, r = i, i + p
        return [P[2 * tout[x] - size[x] + 2] >= size[x] for x in range(n)]

for parent, s, exp in examples:
    got = Solution().findAnswer(parent, s)
    print(got, 'OK' if got == exp else 'SAI')

[True, True, False, True, True, True] OK
[True, True, True, True, True] OK


Với `l = tout − size + 1`, `r = tout`: tâm `l + r + 1 = 2·tout − size + 2`, độ dài `size`.

## 10. Vì sao thuật toán đúng

1. **Đoạn liên tiếp:** trong `dfs(0)`, lời gọi `dfs(x)` ghi ra đúng các ký tự của cây con `x` theo hậu thứ tự, liên tiếp nhau (các lời gọi lồng nhau không chen ký tự ngoài cây con vào giữa), và ký tự cuối là `s[x]`. Thứ tự duyệt con trong `dfs(x)` độc lập với lời gọi bên ngoài ⇒ chuỗi `dfs(x)` khi gọi riêng **bằng** đoạn `T[tout[x] − size[x] + 1 .. tout[x]]`.
2. **Manacher:** `P[c]` là bán kính palindrome cực đại tại tâm `c` của `t`; một đoạn độ dài `L` có tâm `c` là palindrome ⇔ `L ≤ P[c]` (palindrome lồng nhau cùng tâm).
3. **Ngăn xếp lặp** mô phỏng đúng đệ quy: con được đẩy theo thứ tự tăng, nút được thăm sau khi mọi con đã xong.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n · depth)` (tới `n²`) | `O(n)` |
| 2. Băm 2 modulo | `O(n)` | `O(n)` |
| 3. Manacher | `O(n)` | `O(n)` |
| 4. So chuỗi trên `T` | `O(n · depth)` (hằng số nhỏ) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'hash': hash_solution, 'manacher': manacher_solution, 'slice': slice_solution,
         'submit': lambda p, s: Solution().findAnswer(p, s)}
edge = [
    ([-1], "z", [True]),
    ([-1, 0], "ab", [False, True]),
    ([-1, 0], "aa", [True, True]),
    ([-1, 0, 1, 2], "abba", [True, False, False, True]),      # đường thẳng: T = s[3] s[2] s[1] s[0]
    ([-1, 0, 0], "bab", [False, True, True]),                  # T = s1 s2 s0 = "abb"
]
for parent, s, exp in examples + edge:
    assert brute(parent, s) == exp, (parent, s, brute(parent, s))
    for name, f in impls.items():
        assert f(parent, s) == exp, (name, parent, s)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3327)
def random_tree(n):
    return [-1] + [rng.randrange(i) for i in range(1, n)]

cnt = 0
for _ in range(500):
    n = rng.randint(1, 12)
    parent = random_tree(n)
    s = ''.join(rng.choice('ab') for _ in range(n))
    exp = brute(parent, s)
    for name, f in impls.items():
        assert f(parent, s) == exp, (name, parent, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 500 cây ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
import time
N = 10 ** 5
big = {
    'cây ngẫu nhiên': (random_tree(N), ''.join(rng.choice('ab') for _ in range(N))),
    'đường thẳng (sâu 1e5)': ([-1] + list(range(N - 1)), 'a' * N),
    'hình sao': ([-1] + [0] * (N - 1), ''.join(rng.choice('ab') for _ in range(N))),
    'cây nhị phân': ([-1] + [(i - 1) // 2 for i in range(1, N)], 'a' * N),
}
for name, (parent, s) in big.items():
    t = time.time(); a = Solution().findAnswer(parent, s); t1 = time.time() - t
    t = time.time(); b = hash_solution(parent, s); t2 = time.time() - t
    assert a == b
    print(f"{name:22s}: {sum(a):6d} true | nộp {t1:.2f}s | băm {t2:.2f}s")

cây ngẫu nhiên        :  65368 true | nộp 0.21s | băm 0.35s


đường thẳng (sâu 1e5) : 100000 true | nộp 0.15s | băm 0.25s


hình sao              :  99999 true | nộp 0.15s | băm 0.27s


cây nhị phân          : 100000 true | nộp 0.13s | băm 0.28s


Kiểm tra độc lập: chuỗi toàn `'a'` ⇒ mọi đáp án `true`; hình sao ⇒ mọi lá `true`, gốc `true` ⇔ `s[1..n−1] + s[0]` là palindrome.

In [13]:
assert all(Solution().findAnswer([-1] + list(range(N - 1)), 'a' * N))
parent, s = big['hình sao']
ans = Solution().findAnswer(parent, s)
w = s[1:] + s[0]
assert all(ans[1:]) and ans[0] == (w == w[::-1])
print('đường thẳng toàn a và hình sao: khớp.')

đường thẳng toàn a và hình sao: khớp.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng tiền thứ tự (thêm `s[x]` trước khi duyệt con)

In [14]:
def wrong_preorder(parent, s):
    ch = build_children(parent)
    res = []
    for i in range(len(parent)):
        out, st = [], [i]
        while st:
            x = st.pop()
            out.append(s[x])                       # SAI: thêm trước con
            st.extend(reversed(ch[x]))
        w = ''.join(out); res.append(w == w[::-1])
    return res

print('sai :', wrong_preorder(*examples[0][:2]), '\nđúng:', Solution().findAnswer(*examples[0][:2]))

sai : [False, False, False, True, True, True] 
đúng: [True, True, False, True, True, True]


**Vì sao sai:** đề thêm `s[x]` **sau** khi duyệt hết con (hậu thứ tự). Với ví dụ 1, tiền thứ tự cho `dfs(0) = s0 s1 s3 s4 s2 s5 = "aabaab"` (không palindrome) và `dfs(1) = "aab"`, trong khi chuỗi đúng là `"abaaba"` và `"aba"` — hai nút đầu bị trả lời sai.

### Lỗi 2 — Duyệt con theo thứ tự giảm dần

In [15]:
def wrong_child_order(parent, s):
    n = len(parent)
    ch = build_children(parent)
    ch = [c[::-1] for c in ch]                     # SAI: con theo thứ tự giảm
    T, tout, size, st = [], [0] * n, [1] * n, [(0, 0)]
    while st:
        x, i = st.pop()
        if i < len(ch[x]):
            st.append((x, i + 1)); st.append((ch[x][i], 0))
        else:
            tout[x] = len(T); T.append(s[x])
            if x:
                size[parent[x]] += size[x]
    T = ''.join(T)
    return [T[tout[x] - size[x] + 1: tout[x] + 1] == T[tout[x] - size[x] + 1: tout[x] + 1][::-1] for x in range(n)]

bad = ([-1, 0, 0, 1], "abba")
print('sai :', wrong_child_order(*bad), '| đúng:', Solution().findAnswer(*bad), '| vét cạn:', brute(*bad))

sai : [False, False, True, True] | đúng: [True, False, True, True] | vét cạn: [True, False, True, True]


**Vì sao sai:** gốc 0 có con `1` (cây con `{1, 3}`) và `2`. Đúng thứ tự: `s3 s1 s2 s0 = "abba"` (palindrome); bản sai cho `s2 s3 s1 s0 = "baba"`. Thứ tự con làm thay đổi chuỗi.

### Lỗi 3 — Sai chỉ số tâm Manacher

In [16]:
def wrong_center(parent, s):
    T, tout, size = postorder(parent, s)
    P = manacher(T)
    return [P[(tout[x] - size[x] + 1) + tout[x]] >= size[x] for x in range(len(parent))]   # SAI: thiếu +1

print('sai :', wrong_center(*examples[0][:2]), '\nđúng:', Solution().findAnswer(*examples[0][:2]))

sai : [False, False, False, False, False, True] 
đúng: [True, True, False, True, True, True]


**Vì sao sai:** ký tự `T[i]` nằm ở vị trí `2i + 1` trong `t`, nên tâm của đoạn `[l, r]` là `(2l + 1 + 2r + 1)/2 = l + r + 1`.

### Lỗi 4 — DFS đệ quy trên cây sâu

In [17]:
import sys
def recursive_T(parent, s):
    ch = build_children(parent)
    out = []
    def dfs(x):
        for y in ch[x]:
            dfs(y)
        out.append(s[x])
    dfs(0)
    return ''.join(out)

old = sys.getrecursionlimit()
try:
    recursive_T([-1] + list(range(N - 1)), 'a' * N)
    print('không lỗi (giới hạn đệ quy đủ lớn)')
except RecursionError as e:
    print('RecursionError:', str(e)[:40], '... -> cần duyệt lặp bằng ngăn xếp')
finally:
    sys.setrecursionlimit(old)

RecursionError: maximum recursion depth exceeded ... -> cần duyệt lặp bằng ngăn xếp


**Vì sao sai:** Python giới hạn độ sâu đệ quy (~1000 mặc định); cây đường thẳng `10^5` nút làm tràn. Tăng `setrecursionlimit` có thể làm sập tiến trình (tràn stack C) — duyệt lặp là cách an toàn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi/mảng sinh ra bởi DFS trên **từng cây con**" ⇒ cây con = đoạn liên tiếp trong thứ tự Euler (tiền hoặc hậu thứ tự) ⇒ bài toán truy vấn đoạn.
- "Đoạn con có là palindrome?" nhiều lần ⇒ Manacher (chắc chắn) hoặc băm (đơn giản, xác suất).
- Cây `10^5` nút ⇒ luôn chuẩn bị duyệt lặp.

## 15. Biến thể của bài

1. Dùng **tiền thứ tự** (thêm `s[x]` trước) ⇒ đoạn `[tin[x], tin[x] + size[x] − 1]`.
2. Hỏi **độ dài palindrome dài nhất** trong chuỗi của mỗi cây con ⇒ khó hơn (cần truy vấn trên đoạn).
3. Cập nhật ký tự của một nút rồi hỏi lại ⇒ băm + cây Fenwick/segment tree.
4. Kiểm tra chuỗi cây con có thể **hoán vị** thành palindrome ⇒ chỉ cần số ký tự lẻ ≤ 1 (DSU on tree / XOR bitmask theo đoạn).

In [18]:
# Biến thể 1 và 4
def preorder_variant(parent, s):
    n = len(parent)
    ch = build_children(parent)
    T, tin, size = [], [0] * n, [1] * n
    order, st = [], [0]
    while st:
        x = st.pop()
        tin[x] = len(T); T.append(s[x]); order.append(x)
        st.extend(reversed(ch[x]))
    for x in reversed(order):
        if x:
            size[parent[x]] += size[x]
    P = manacher(''.join(T))
    return [is_pal(P, tin[x], tin[x] + size[x] - 1) for x in range(n)]

def permutation_palindrome(parent, s):
    T, tout, size = postorder(parent, s)
    pre = [0]
    for c in T:
        pre.append(pre[-1] ^ (1 << (ord(c) - 97)))
    return [bin(pre[tout[x] + 1] ^ pre[tout[x] - size[x] + 1]).count('1') <= 1 for x in range(len(parent))]

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 10); parent = random_tree(n); s = ''.join(rng.choice('abc') for _ in range(n))
    assert preorder_variant(parent, s) == wrong_preorder(parent, s)        # "sai" của bài gốc = đúng của biến thể 1
    ch = build_children(parent)
    for x in range(n):
        out, st = [], [x]
        while st:
            y = st.pop(); out.append(s[y]); st.extend(ch[y])
        odd = sum(1 for c in set(out) if out.count(c) % 2)
        assert permutation_palindrome(parent, s)[x] == (odd <= 1)
print('Biến thể 1 (tiền thứ tự) và 4 (hoán vị thành palindrome) khớp vét cạn.')

Biến thể 1 (tiền thứ tự) và 4 (hoán vị thành palindrome) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Gọi `dfs(i)` từng nút | `O(n · depth)` | Oracle |
| 2 | Đoạn con + băm | `O(n)` | Xác suất, dễ viết |
| 3 | Đoạn con + Manacher | `O(n)` | **Nộp bài**, chắc chắn |
| 4 | Đoạn con + so chuỗi | `O(n · depth)` | Hằng số nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `parent = [-1, 0, 1]`, `s = "aba"`.

**Bài 2 (tính tay):** `parent = [-1, 0, 0, 0]`, `s = "xabc"`.

**Bài 3 (điền TODO):** trả về luôn chuỗi `dfs(x)` của mọi `x` dưới dạng cặp chỉ số `(l, r)` trong `T`.

**Bài 4 (tự viết lại):** đếm số nút có chuỗi cây con là palindrome **độ dài chẵn**.

**Bài 5 (mini challenge):** biến thể 3 — hỗ trợ cập nhật `s[x] = c` và truy vấn `isPal(x)` (băm + Fenwick).

In [19]:
def intervals_todo(parent, s):
    # TODO
    raise NotImplementedError

def even_pal_count_todo(parent, s):
    # TODO
    raise NotImplementedError

class DynamicTreePal_todo:
    def __init__(self, parent, s):
        # TODO
        raise NotImplementedError

print('Đã định nghĩa các hàm/lớp TODO (không gọi).')

Đã định nghĩa các hàm/lớp TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường thẳng `0 → 1 → 2`. (2) `T = s2 s1 s0 = "aba"`, nút 1: `"ba"`. (3) `[True, False, True]`.

**Bài 2.** (1) Sao: `T = "abcx"`. (2) Gốc: `"abcx"` ✗, lá ✓. (3) `[False, True, True, True]`.

**Bài 3.** (1) `r = tout[x]`, `l = r − size[x] + 1`. (2) Một lần hậu thứ tự. (3) Kiểm bằng vét cạn.

**Bài 4.** (1) Lọc các nút `size[x]` chẵn. (2) Dùng Manacher. (3) Đếm.

**Bài 5.** (1) Hậu thứ tự cố định khi ký tự đổi (cây không đổi). (2) Fenwick lưu `c_i · B^i` cho chuỗi xuôi và `c_i · B^{n−1−i}` cho chuỗi ngược. (3) So hai hash đoạn sau khi chuẩn hóa lũy thừa.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3, 4
assert Solution().findAnswer([-1, 0, 1], "aba") == [True, False, True]
assert Solution().findAnswer([-1, 0, 0, 0], "xabc") == [False, True, True, True]

def intervals(parent, s):
    T, tout, size = postorder(parent, s)
    return T, [(tout[x] - size[x] + 1, tout[x]) for x in range(len(parent))]

def even_pal_count(parent, s):
    T, tout, size = postorder(parent, s)
    P = manacher(T)
    return sum(1 for x in range(len(parent)) if size[x] % 2 == 0 and is_pal(P, tout[x] - size[x] + 1, tout[x]))

rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 10); parent = random_tree(n); s = ''.join(rng.choice('ab') for _ in range(n))
    T, iv = intervals(parent, s)
    ch = build_children(parent)
    exp_even = 0
    for x in range(n):
        out, st = [], [(x, 0)]
        while st:
            y, j = st.pop()
            if j < len(ch[y]):
                st.append((y, j + 1)); st.append((ch[y][j], 0))
            else:
                out.append(s[y])
        w = ''.join(out)
        assert T[iv[x][0]: iv[x][1] + 1] == w
        exp_even += len(w) % 2 == 0 and w == w[::-1]
    assert even_pal_count(parent, s) == exp_even
print('Bài 1, 2 đúng | Bài 3, 4 khớp vét cạn.')

Bài 1, 2 đúng | Bài 3, 4 khớp vét cạn.


In [21]:
# Bài 5
class Fenwick:
    def __init__(self, n, M):
        self.n, self.M, self.t = n, M, [0] * (n + 1)
    def add(self, i, v):
        i += 1
        while i <= self.n:
            self.t[i] = (self.t[i] + v) % self.M; i += i & -i
    def pre(self, i):
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s % self.M

class DynamicTreePal:
    M, B = (1 << 61) - 1, 911382323
    def __init__(self, parent, s):
        self.T, tout, size = postorder(parent, s)
        self.T = list(self.T)
        self.n = n = len(parent)
        self.pos = tout; self.size = size
        self.pw = [1] * (n + 1)
        for i in range(n):
            self.pw[i + 1] = self.pw[i] * self.B % self.M
        self.f, self.g = Fenwick(n, self.M), Fenwick(n, self.M)
        for i, c in enumerate(self.T):
            self._put(i, ord(c), 1)
    def _put(self, i, v, sign):
        self.f.add(i, sign * v * self.pw[i] % self.M)
        self.g.add(i, sign * v * self.pw[self.n - 1 - i] % self.M)
    def update(self, x, c):
        i = self.pos[x]
        self._put(i, ord(self.T[i]), -1); self.T[i] = c; self._put(i, ord(c), 1)
    def is_pal(self, x):
        r = self.pos[x]; l = r - self.size[x] + 1
        a = (self.f.pre(r + 1) - self.f.pre(l)) % self.M      # Σ c_i B^i
        b = (self.g.pre(r + 1) - self.g.pre(l)) % self.M      # Σ c_i B^{n-1-i}
        # palindrome ⇔ a · B^{n-1-r} == b · B^{l}
        return a * self.pw[self.n - 1 - r] % self.M == b * self.pw[l] % self.M

rng = random.Random(4)
for _ in range(100):
    n = rng.randint(1, 10); parent = random_tree(n); s = list(''.join(rng.choice('ab') for _ in range(n)))
    D = DynamicTreePal(parent, ''.join(s))
    for _ in range(10):
        x = rng.randrange(n); c = rng.choice('ab')
        D.update(x, c); s[x] = c
        assert [D.is_pal(y) for y in range(n)] == brute(parent, ''.join(s))
print('Bài 5: cập nhật + truy vấn khớp vét cạn.')

Bài 5: cập nhật + truy vấn khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
children theo thứ tự tăng; hậu thứ tự lặp -> T, tout[x], size[x]
P = manacher('#' + '#'.join(T) + '#')
answer[x] = P[2*tout[x] - size[x] + 2] >= size[x]
```

**Checklist:**

- [ ] Hậu thứ tự (con trước, nút sau)?
- [ ] Con theo thứ tự số hiệu **tăng**?
- [ ] Duyệt lặp, không đệ quy?
- [ ] Tâm Manacher `l + r + 1` trong chuỗi có `#`?
- [ ] `size[parent[x]] += size[x]` khi thăm `x` (con thăm trước cha)?